# Porównawcza analiza skuteczności algorytmów uczenia maszynowego w zadaniu klasyfikacji binarnej zadowolenia pasażerów linii lotniczych na podstawie zbioru danych Airline Passenger Satisfaction.

**Dominika Boguszewska, Natalia Pieczko** — Zaawansowane Uczenie Maszynowe, czerwiec 2026

> **Tryb fast** — ten notebook jest skróconą wersją `main.ipynb`. Zbiór treningowy jest ograniczony do 10 000 próbek, a CV do 3 foldów, co pozwala na szybkie sprawdzenie wyników kosztem precyzji. Wyniki mogą nieznacznie odbiegać od pełnej analizy.

## Konfiguracja środowiska (Google Colab)

Poniższe komórki konfigurują środowisko. Uruchom je kolejno przed właściwą analizą. Lokalnie (poza Colabem) zostaną automatycznie pominięte.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_path = "/content/Airline_Passenger_Satisfaction_Classification"
    if not os.path.exists(repo_path):
        os.system(
            "git clone https://github.com/nataliap203/Airline_Passenger_Satisfaction_Classification.git "
            + repo_path
        )
    os.chdir(repo_path + "/notebooks")
    print("Katalog roboczy:", os.getcwd())

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "scikit-learn>=1.8.0",
        "xgboost>=3.2.0",
        "tabpfn>=8.0.3",
        "matplotlib>=3.10.9",
        "numpy>=2.4.5",
        "pandas",
        "kaggle",
        "pandas-stubs",
    ])
    print("Zależności zainstalowane.")

In [ ]:
if IN_COLAB:
    from pathlib import Path
    from google.colab import files as colab_files

    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_json.exists():
        print("Prześlij plik kaggle.json")
        print("Pobierz z: kaggle.com/settings → API → Create New Token")
        uploaded = colab_files.upload()
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        for _, content in uploaded.items():
            with open(kaggle_dir / "kaggle.json", "wb") as f:
                f.write(content)
        kaggle_json.chmod(0o600)
        print("Dane uwierzytelniające Kaggle zapisane.")
    else:
        print("kaggle.json już istnieje.")

## 1. Pobranie danych i import bibliotek

In [ ]:
import kaggle
from pathlib import Path

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    "teejmahal20/airline-passenger-satisfaction",
    path=DATA_DIR,
    unzip=True,
)

print("Pobrane pliki:")
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
import os
import sys
import pandas as pd
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning:pkg_resources"

sys.path.append("..")

from sklearn.feature_selection import f_classif, mutual_info_classif

from src.preprocessing import (
    load_data,
    analyze_missing_values,
    clean_data,
    build_preprocessor,
    plot_correlation_matrix,
    plot_target_correlation,
    engineer_features,
    split_X_y,
)
from src.evaluation import evaluate, cross_validate_model, plot_pr_curve, plot_roc_curve
from src.models.xgboost_model import tune_hyperparameters
from src.models.logistic_regression import tune_hyperparameters as tune_hyperparameters_lr
from src.models.random_forest import tune_hyperparameters as tune_hyperparameters_rf
from src.scripts.run_selectkbest_experiment import (
    run_selectkbest_experiment,
    get_feature_scores,
    plot_k_vs_metrics,
    plot_feature_scores,
)
from src.scripts.run_rfe_experiment import (
    run_rfe_experiment,
    get_feature_rankings,
    plot_n_features_vs_metrics,
    plot_feature_rankings,
)

## 2. Przygotowanie danych

### 2.1 Ładowanie danych i analiza brakujących wartości

In [ ]:
train, test = load_data()

print("=== Train ===")
print(train.shape)
train.head()

In [ ]:
print("=== Braki w train ===")
analyze_missing_values(train)

print("\n=== Braki w test ===")
analyze_missing_values(test)

### 2.2 Czyszczenie danych

Braki wykryto wyłącznie w `Arrival Delay in Minutes` (~0,3% obserwacji). Zastosowano usunięcie wierszy z brakującymi wartościami.

In [ ]:
train_clean = clean_data(train)
test_clean = clean_data(test)

print("Train po czyszczeniu:", train_clean.shape)
print("Test po czyszczeniu: ", test_clean.shape)

### 2.3 Analiza korelacji

Macierz korelacji identyfikuje zależności między cechami numerycznymi. Analiza korelacji z zmienną docelową wskazuje najsilniejsze predykatory satysfakcji — wyniki stanowią podstawę inżynierii cech.

In [ ]:
plot_correlation_matrix(train_clean)

In [ ]:
plot_target_correlation(train_clean)

### 2.4 Inżynieria cech

Na podstawie analizy danych stworzono dwie nowe cechy agregujące:
- **`total_delay`** — suma opóźnienia przylotu i wylotu (łączne doświadczenie opóźnienia)
- **`avg_service_score`** — średnia ocen 14 usług pokładowych (okazuje się najsilniejszym predyktorem)

In [ ]:
train_feat = engineer_features(train_clean)
test_feat = engineer_features(test_clean)

print("Nowe kolumny:", [c for c in train_feat.columns if c not in train_clean.columns])
train_feat[["Departure Delay in Minutes", "Arrival Delay in Minutes", "total_delay", "avg_service_score"]].describe()

### 2.5 Podział danych i preprocessing

Dane podzielono zgodnie z oryginalnym podziałem Kaggle: ~104 000 próbek treningowych, ~26 000 testowych. Pipeline preprocessingu (fitowany wyłącznie na zbiorze treningowym) obejmuje standaryzację cech numerycznych i one-hot encoding zmiennych kategorycznych. Łącznie uzyskano 29 cech po kodowaniu.

> **Tryb fast:** zbiór treningowy zostaje dodatkowo zredukowany do 10 000 losowych próbek (stratyfikacja), a CV ograniczone do 3 foldów — wyłącznie w celu skrócenia czasu obliczeń.

In [ ]:
X_train, y_train = split_X_y(train_feat)
X_test, y_test = split_X_y(test_feat)

preprocessor = build_preprocessor(X_train)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("Rozkład klas y_train:\n", y_train.value_counts())

In [ ]:
# Fast mode: ograniczamy zbiór treningowy do 10 000 próbek
from sklearn.model_selection import train_test_split as _tts
X_train, _, y_train, _ = _tts(
    X_train, y_train, train_size=10_000, stratify=y_train, random_state=42
)
preprocessor = build_preprocessor(X_train)
print(f"Fast mode — X_train zredukowany do {X_train.shape}")

## 3. Eksperyment 1 — Porównanie modeli klasycznych i zespołowych

**Pytanie badawcze:** Jak duży przyrost jakości predykcji oferują modele zespołowe w porównaniu do regresji logistycznej?

**Hipoteza:** XGBoost, dzięki technice gradient boosting, osiągnie najwyższe wyniki na pełnym zbiorze danych.

Każdy model poddano strojeniu hiperparametrów metodą GridSearchCV (3-krotna CV w trybie fast, kryterium: F1). Ocena końcowa na wydzielonym zbiorze testowym, uzupełniona krzywą Precision-Recall i krzywą ROC.

### 3.1 Regresja logistyczna (baseline)

In [ ]:
search = tune_hyperparameters_lr(preprocessor, X_train, y_train, cv=3)

print("Najlepsze parametry:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nNajlepsze F1 (CV): {search.best_score_:.4f}")

In [ ]:
cv_scores = cross_validate_model(search.best_estimator_, X_train, y_train, cv=3)

print("Walidacja krzyżowa najlepszego modelu (3-fold):")
print(cv_scores.to_string(index=False))
print("\nŚrednie:")
print(cv_scores.mean().map("{:.4f}".format).to_string())
print("\nOdchylenia standardowe:")
print(cv_scores.std().map("{:.4f}".format).to_string())

In [ ]:
best_model = search.best_estimator_
metrics_lr = evaluate(best_model, X_test, y_test)

In [ ]:
plot_pr_curve(best_model, X_test, y_test, label="Regresja logistyczna")

In [ ]:
plot_roc_curve(best_model, X_test, y_test, label="Regresja logistyczna")

### 3.2 Las losowy

In [ ]:
search = tune_hyperparameters_rf(preprocessor, X_train, y_train, cv=3)

print("Najlepsze parametry:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nNajlepsze F1 (CV): {search.best_score_:.4f}")

In [ ]:
cv_scores = cross_validate_model(search.best_estimator_, X_train, y_train, cv=3)

print("Walidacja krzyżowa najlepszego modelu (3-fold):")
print(cv_scores.to_string(index=False))
print("\nŚrednie:")
print(cv_scores.mean().map("{:.4f}".format).to_string())
print("\nOdchylenia standardowe:")
print(cv_scores.std().map("{:.4f}".format).to_string())

In [ ]:
best_model = search.best_estimator_
metrics_rf = evaluate(best_model, X_test, y_test)

In [ ]:
plot_pr_curve(best_model, X_test, y_test, label="Las losowy")

In [ ]:
plot_roc_curve(best_model, X_test, y_test, label="Las losowy")

### 3.3 XGBoost

In [ ]:
search = tune_hyperparameters(preprocessor, X_train, y_train, cv=3)

print("Najlepsze parametry:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nNajlepsze F1 (CV): {search.best_score_:.4f}")

In [ ]:
cv_scores = cross_validate_model(search.best_estimator_, X_train, y_train, cv=3)

print("Walidacja krzyżowa najlepszego modelu (3-fold):")
print(cv_scores.to_string(index=False))
print("\nŚrednie:")
print(cv_scores.mean().map("{:.4f}".format).to_string())
print("\nOdchylenia standardowe:")
print(cv_scores.std().map("{:.4f}".format).to_string())

In [ ]:
best_model = search.best_estimator_
metrics_xgb = evaluate(best_model, X_test, y_test)

In [ ]:
plot_pr_curve(best_model, X_test, y_test, label="XGBoost")

In [ ]:
plot_roc_curve(best_model, X_test, y_test, label="XGBoost")

### 3.4 Wnioski z Eksperymentu 1

Modele zespołowe (Las Losowy, XGBoost) przewyższają Regresję Logistyczną o ~10 punktów procentowych F1, potwierdzając hipotezę. XGBoost i Las Losowy osiągają zbliżone wyniki (F1 ~0.96, ROC AUC ~0.995), co wskazuje na zdolność obu podejść do uchwycenia nieliniowych zależności w zbiorze. Regresja logistyczna, mimo niższych metryk, dostarcza interpretowalnych wag cech i stanowi użyteczny punkt odniesienia.

In [ ]:
results_exp1 = (
    pd.DataFrame(
        {"Regresja logistyczna": metrics_lr, "Las losowy": metrics_rf, "XGBoost": metrics_xgb}
    )
    .T.rename(columns={"accuracy": "Accuracy", "f1": "F1", "roc_auc": "ROC AUC"})
)
results_exp1.index.name = "Model"
results_exp1.round(4).style.highlight_max(axis=0, props="font-weight: bold; color: green")

## 4. Eksperyment 2 — Efektywność próbkowania z wykorzystaniem TabPFN

**Pytanie badawcze:** Czy pretrenowany model oparty na architekturze Transformer (TabPFN) potrafi osiągnąć lepsze wyniki generalizacji na bardzo małym zbiorze treningowym w porównaniu do klasycznych metod uczenia maszynowego?

**Hipoteza:** Przy mocno ograniczonej liczbie próbek treningowych TabPFN osiągnie wyższą skuteczność niż XGBoost i Las Losowy trenowane na tych samych danych.

Eksperyment porównuje trzy modele trenowane na losowych podzbiorach treningowych o rozmiarach [100, 250, 500] i ewaluowane na pełnym zbiorze testowym. Modele klasyczne używają najlepszych hiperparametrów znalezionych w Eksperymencie 1. W odróżnieniu od Eksperymentu 1, walidacja krzyżowa nie jest stosowana — celem jest ocena jakości przy dokładnie N próbkach treningowych, a nie szacowanie wariancji modelu.

In [ ]:
import subprocess, sys, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
script = "src/scripts/run_tabpfn_experiment.py"
with subprocess.Popen(
    [sys.executable, "-u", str(script)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd="..",
) as proc:
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"\nBłąd: skrypt zakończył się kodem {proc.returncode}")

In [ ]:
results_df = pd.read_csv("../data/tabpfn_experiment_results.csv").set_index(["model", "sample_size"])
print(results_df.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["accuracy", "f1", "roc_auc"]):
    for model_name, group in results_df.reset_index().groupby("model"):
        ax.plot(group["sample_size"], group[metric], marker="o", label=model_name)
    ax.set_title(metric)
    ax.set_xlabel("Liczba próbek treningowych")
    ax.set_ylabel(metric)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("TabPFN vs XGBoost - wpływ liczby próbek treningowych")
plt.tight_layout()
plt.show()

### 4.1 Wnioski z Eksperymentu 2

Hipoteza potwierdzona: TabPFN systematycznie przewyższa XGBoost i Las Losowy przy każdym rozmiarze próbki, demonstrując wyższą efektywność próbkowania. Różnica maleje wraz ze wzrostem rozmiaru próbki — korzyść z pretrainingu zmniejsza się, gdy dostępnych jest więcej danych. Wynik potwierdza przydatność modeli pretrenowanych w scenariuszach z deficytem danych treningowych.

## 5. Eksperyment 3 — Analiza ważności cech i selekcja

**Pytanie badawcze:** Które z cech lotu i pasażera są najsilniejszymi predykatorami zadowolenia i w jakim stopniu różne metody selekcji cech (SelectKBest, RFE) są ze sobą zgodne w ich identyfikacji?

**Hipoteza:** Oceny usług pokładowych będą dominującymi predyktorami. Rankingi cech uzyskane różnymi metodami będą spójne, a ich selekcja pozwoli uprościć model regresji logistycznej bez pogorszenia jego skuteczności.

### 5.1 SelectKBest

Metoda filtracyjna; wybiera `k` najlepszych cech na podstawie testu statystycznego. Porównano dwie funkcje oceny: `f_classif` (test ANOVA, zależności liniowe) i `mutual_info_classif` (wzajemna informacja, zależności nieliniowe).

In [ ]:
results = run_selectkbest_experiment(
    preprocessor, X_train, y_train, X_test, y_test,
    k_values=[5, 10, 15, 20, 25, "all"],
    score_funcs=[f_classif, mutual_info_classif]
)

In [ ]:
print(results.to_string(index=False))

In [ ]:
plot_k_vs_metrics(results)

In [ ]:
scores_f = get_feature_scores(preprocessor, X_train, y_train, f_classif)
scores_f

In [ ]:
plot_feature_scores(scores_f,  top_n=29)

In [ ]:
scores_mi = get_feature_scores(preprocessor, X_train, y_train, mutual_info_classif)
scores_mi

In [ ]:
plot_feature_scores(scores_mi, top_n=29)

### 5.2 Wnioski z SelectKBest

Jakość modelu rośnie gwałtownie do k ≈ 20, po czym osiąga plateau — dalsze dodawanie cech przynosi niewielki wzrost F1. Rankingi `f_classif` i `mutual_info_classif` wykazują dużą zgodność w identyfikacji najważniejszych cech, co potwierdza spójność obu podejść. Cechy `avg_service_score` oraz `online boarding` znajdują się w czołówce obu rankingów, potwierdzając ich silną predykcyjność. Warto zauważyć, że występują różnice w istotności cech pomiędzy modelami.

### 5.3 RFE (Recursive Feature Elimination)

Metoda iteracyjnie eliminuje najmniej istotne cechy na podstawie wag modelu. Porównano dwa estymatory bazowe: Regresję Logistyczną i Las Losowy.

In [ ]:
# Fast mode: zmniejszamy liczbę drzew RF używanych w RFE
from sklearn.ensemble import RandomForestClassifier as _RFC
from sklearn.linear_model import LogisticRegression as _LR

_fast_rfe_estimators = {
    "LR": _LR(C=0.01, l1_ratio=1.0, solver="saga", max_iter=5000, random_state=42),
    "RF": _RFC(n_estimators=20, random_state=42, n_jobs=1),
}

In [ ]:
results_rfe = run_rfe_experiment(
    preprocessor, X_train, y_train, X_test, y_test,
    estimators=_fast_rfe_estimators,
)

In [ ]:
print(results_rfe.to_string(index=False))

In [ ]:
plot_n_features_vs_metrics(results_rfe)

In [ ]:
rankings_lr = get_feature_rankings(
    preprocessor, X_train, y_train,
    estimator_name="LR",
    estimators=_fast_rfe_estimators,
)
rankings_lr

In [ ]:
plot_feature_rankings(rankings_lr, estimator_name="LR", top_n=29)

In [ ]:
rankings_rf = get_feature_rankings(
    preprocessor, X_train, y_train,
    estimator_name="RF",
    estimators=_fast_rfe_estimators,
)
rankings_rf

In [ ]:
plot_feature_rankings(rankings_rf, estimator_name="RF", top_n=29)

### 5.4 Wnioski z RFE

Rankingi cech uzyskane metodą RFE dla regresji logistycznej i lasu losowego różnią się szczególnie pod tym względem, że RFE dla regresji logistycznej nie wskazuje na `avg_service_score` jako jedną z najważniejszych cech, podczas gdy RFE dla lasu losowego umieszcza ją dość wysoko (podobnie jak metoda SelectKBest). RFE dla lasu losowego wykazuje większą zgodność z wynikami SelectKBest, co może wynikać z podobnej zdolności do uchwycenia nieliniowych zależności. Selekcja ~20–25 cech pozwala zachować pełną jakość modelu LR, potwierdzając hipotezę Eksperymentu 3.

## 6. Wnioski końcowe

| Aspekt | Wniosek |
|---|---|
| **Najlepszy model (pełne dane)** | XGBoost i Las Losowy — zbliżone wyniki (F1 ≈ 0.96, ROC AUC ≈ 0.995); XGBoost nieznacznie lepszy |
| **Hipoteza Eksperymentu 1** | ✓ Potwierdzona — XGBoost najlepszy na pełnych danych |
| **Hipoteza Eksperymentu 2** | ✓ Potwierdzona — TabPFN lepszy przy małych próbkach (100–500) |
| **Hipoteza Eksperymentu 3** | ✓ Potwierdzona — usługi pokładowe dominują; k ≈ 20 wystarczy dla regresji logistycznej |
| **Najważniejsze cechy** | `avg_service_score`, `Online boarding`, klasa podróży (`Business`/`Eco`) |
| **Zgodność SelectKBest i RFE** | Wysoka — obie metody wskazują ten sam zestaw kluczowych predyktorów |

**Rekomendacja praktyczna:**
- Do produkcji z pełnymi danymi: **XGBoost** lub **Las Losowy** (najwyższa jakość)
- Do zastosowań z deficytem danych treningowych: **TabPFN** (wyższa efektywność próbkowania)
- Do szybkiego, interpretowalnego modelu z selekcją cech: **Regresja logistyczna + SelectKBest (k=20)**

In [ ]:
results_exp1.round(4).style.highlight_max(axis=0, props="font-weight: bold; color: green")